In [ ]:
# ============================================================
# 10 — LEADERBOARD SUBMISSION (EB-NeRD)
# Load trained LightGBM, encode testset with multilingual MiniLM, predict on 13.5M -> zip.
# Fully self-contained EB-NeRD notebook. Hardcoded paths.
# ============================================================
!pip install lightgbm sentence-transformers polars -q
import os, glob, math, zipfile, numpy as np, polars as pl, datetime as dt, lightgbm as lgb, warnings
warnings.filterwarnings("ignore")
from bisect import bisect_left
from collections import defaultdict, Counter
# ---- hardcoded EB-NeRD large + testset (double-nested!) paths ----
LARGE = "/kaggle/input/datasets/wrathofgod123/ebnerd-complete/ebnerd_large"
TEST  = "/kaggle/input/datasets/wrathofgod123/ebnerd-complete/ebnerd_testset/ebnerd_testset"
MODEL = "/kaggle/input/datasets/wrathofgod123/trained-model/ebnerd_large_lgbm.txt"
print("LARGE exists:", os.path.exists(f"{LARGE}/articles.parquet"))
print("TEST  exists:", os.path.exists(f"{TEST}/articles.parquet"))
PREFIX = "eb"
def pfx(x): return f"{PREFIX}:{x}"
def _prefix(col): return pl.concat_str([pl.lit(f"{PREFIX}:"), col.cast(pl.Utf8)])


In [ ]:
# ---- load trained model + testset article luts (fresh test-period stats) ----
rk = lgb.Booster(model_file=MODEL)
print("model loaded | features:", rk.num_feature())
at=pl.read_parquet(f"{TEST}/articles.parquet")
pub={pfx(r):p for r,p in zip(at["article_id"].to_list(),at["published_time"].to_list())}
pv ={pfx(r):(v or 0) for r,v in zip(at["article_id"].to_list(),at["total_pageviews"].to_list())}
iv ={pfx(r):(v or 0) for r,v in zip(at["article_id"].to_list(),at["total_inviews"].to_list())}
rt ={pfx(r):(v or 0) for r,v in zip(at["article_id"].to_list(),at["total_read_time"].to_list())}
cat={pfx(r):(c or "") for r,c in zip(at["article_id"].to_list(),at["category_str"].to_list())}
txt={pfx(r):f"{t or ''} {s or ''}".strip() for r,t,s in zip(at["article_id"].to_list(),at["title"].to_list(),at["subtitle"].to_list())}
pv_logmax=np.log1p(max([v for v in pv.values() if v>0] or [1]))
print("testset articles:",len(pub))


In [ ]:
from sentence_transformers import SentenceTransformer
minilm=SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
aids=list(txt.keys()); atxt=[txt[i] if txt[i] else "nyhed" for i in aids]
emb=minilm.encode(atxt,batch_size=512,normalize_embeddings=True,convert_to_numpy=True,show_progress_bar=True)
emb_by_id={aids[i]:emb[i] for i in range(len(aids))}
print("MiniLM encoded")

def recency(aid,T,tau=24.0):
    p=pub.get(aid)
    if p is None or T is None: return 0.0
    dh=(T-p).total_seconds()/3600.0
    return float(np.exp(-dh/tau)) if dh>=0 else 0.0
def load_hist(base):
    h=pl.read_parquet(f"{base}/history.parquet")
    return {pfx(u):[pfx(x) for x in (arts or [])] for u,arts in
            zip(h["user_id"].to_list(), h["article_id_fixed"].to_list())}
def user_prof(hist,mh=30):
    ai=hist[-mh:] if hist else []
    cats=[cat.get(x) for x in ai];tot=len([c for c in cats if c])
    cc=Counter(c for c in cats if c);cd={k:v/tot for k,v in cc.items()} if tot else {}
    hv=[emb_by_id[x] for x in ai if x in emb_by_id]
    um=np.mean(hv,0) if hv else None
    if um is not None: um=um/(np.linalg.norm(um)+1e-9)
    return cd,um,hv
def mm(x):
    lo,hi=x.min(),x.max();return np.zeros_like(x) if hi-lo<1e-12 else (x-lo)/(hi-lo)
def build_feats(uid,T,cand,hl):
    cd,um,hv=user_prof(hl.get(uid,[]));m=len(cand);F=[]
    for i,c in enumerate(cand):
        rec=recency(c,T)
        p=np.log1p(pv.get(c,0))/pv_logmax
        ivn=np.log1p(iv.get(c,0))/pv_logmax
        rtn=np.log1p(rt.get(c,0))/pv_logmax
        ctr=pv.get(c,0)/iv.get(c,1) if iv.get(c,0)>0 else 0.0
        cm=cd.get(cat.get(c,""),0.0)
        cv=emb_by_id.get(c)
        em=float(um@cv) if (um is not None and cv is not None) else 0.0
        eb=float(max((v@cv for v in hv),default=0.0)) if cv is not None else 0.0
        sn=0.0;pos=i/max(1,m-1)
        F.append([rec,p,ivn,rtn,ctr,cm,em,eb,sn,pos,m])
    F=np.array(F); F[:,0]=mm(F[:,0])
    return F


In [ ]:
# ---- predict on the full 13.5M testset, write ranked predictions.txt + zip ----
hist_test=load_hist(f"{TEST}/test")
b_te=pl.read_parquet(f"{TEST}/test/behaviors.parquet",
                     columns=["impression_id","user_id","impression_time","article_ids_inview"])
print("predicting",b_te.height,"test impressions...")
ci=0
with open("/kaggle/working/predictions.txt","w") as fout:
    for iid,uid,T,inv in zip(b_te["impression_id"].to_list(),b_te["user_id"].to_list(),
                             b_te["impression_time"].to_list(),b_te["article_ids_inview"].to_list()):
        cand=[pfx(x) for x in inv]
        sc=rk.predict(build_feats(pfx(uid),T,cand,hist_test))
        order=np.argsort(-sc);ranks=np.empty(len(cand),int);ranks[order]=np.arange(1,len(cand)+1)
        fout.write(f"{iid} [{','.join(map(str,ranks.tolist()))}]\n")
        ci+=1
        if ci%1000000==0: print(f"  {ci}/{b_te.height}")
with zipfile.ZipFile("/kaggle/working/ebnerd_large_submission.zip","w",zipfile.ZIP_DEFLATED) as z:
    z.write("/kaggle/working/predictions.txt","predictions.txt")
print(f"DONE. {ci} impressions -> ebnerd_large_submission.zip")
print("Upload to Codabench RecSys 2024 competition 2469.")
